# EDA — Incidencia Delictiva Municipal (SESNSP 2025)

Análisis exploratorio con gráficas interactivas (Plotly) del dataset `sesnsp_crime_monthly_2025.parquet`.

**Contenido**
1. Carga y panorama general
2. Calidad de datos
3. Distribución de incidencia
4. Análisis temporal por mes
5. Ranking de entidades
6. Top municipios
7. Tipos de delito y bien jurídico
8. Evolución mensual — Top 5 delitos
9. Heatmap entidad × mes
10. Concentración geográfica (Pareto)
11. Homicidios por entidad
12. Estacionalidad por tipo de delito
13. Insights para reporte comercial

In [6]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

MONTH_LABELS = [
    "Ene", "Feb", "Mar", "Abr", "May", "Jun",
    "Jul", "Ago", "Sep", "Oct", "Nov", "Dic",
]
MONTH_ORDER = [
    "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
    "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre",
]

## 1. Carga y panorama general

In [7]:
df = pd.read_parquet("../data/silver/crime_monthly/sesnsp_crime_monthly_2025.parquet")
print(f"Filas: {len(df):,}  |  Columnas: {len(df.columns)}")
print(f"Entidades: {df['entidad'].nunique()} | Municipios: {df['cvegeo'].nunique()} | Tipos de delito: {df['tipo_delito'].nunique()}")
print(f"Meses cubiertos: {sorted(df['mes'].unique())}")
df.head()

Filas: 2,923,536  |  Columnas: 15
Entidades: 32 | Municipios: 2486 | Tipos de delito: 40
Meses cubiertos: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


,anio,fecha,mes,mes_nombre,clave_entidad,cvegeo,entidad,municipio,bien_juridico,tipo_delito,subtipo_delito,modalidad,incidencia,record_source,load_date
0,2025,2025-01-01,1,Enero,01,01001,Aguascalientes,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,4,SESNSP,2026-04-30 03:51:52.817724+00:00
1,2025,2025-01-01,1,Enero,01,01001,Aguascalientes,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,2,SESNSP,2026-04-30 03:51:52.817724+00:00
2,2025,2025-01-01,1,Enero,01,01001,Aguascalientes,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,SESNSP,2026-04-30 03:51:52.817724+00:00
3,2025,2025-01-01,1,Enero,01,01001,Aguascalientes,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,0,SESNSP,2026-04-30 03:51:52.817724+00:00
4,2025,2025-01-01,1,Enero,01,01001,Aguascalientes,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,SESNSP,2026-04-30 03:51:52.817724+00:00


In [8]:
df.dtypes.to_frame("dtype")

,dtype
anio,int64
fecha,datetime64[us]
mes,int64
mes_nombre,str
clave_entidad,str
cvegeo,str
entidad,str
municipio,str
bien_juridico,str
tipo_delito,str


In [9]:
df.describe()

,anio,fecha,mes,incidencia
count,2923536.0,2923536,2.923536e+06,2.923536e+06
mean,2025.0,2025-06-16 12:00:00,6.500000e+00,6.898205e-01
min,2025.0,2025-01-01 00:00:00,1.000000e+00,0.000000e+00
25%,2025.0,2025-03-24 06:00:00,3.750000e+00,0.000000e+00
50%,2025.0,2025-06-16 00:00:00,6.500000e+00,0.000000e+00
75%,2025.0,2025-09-08 12:00:00,9.250000e+00,0.000000e+00
max,2025.0,2025-12-01 00:00:00,1.200000e+01,1.571000e+03
std,0.0,NaN,3.452053e+00,8.292606e+00


## 2. Calidad de datos

In [10]:
nulos = df.isnull().sum()
n_dup = df.duplicated().sum()
n_zero = (df["incidencia"] == 0).sum()
pct_zero = n_zero / len(df) * 100

print(f"Valores nulos totales: {nulos.sum()}")
print(f"Filas duplicadas exactas: {n_dup:,}")
print(f"Filas con incidencia = 0: {n_zero:,} ({pct_zero:.1f}%)")
print(f"Filas con incidencia > 0: {len(df) - n_zero:,} ({100 - pct_zero:.1f}%)")

Valores nulos totales: 0
Filas duplicadas exactas: 0
Filas con incidencia = 0: 2,660,445 (91.0%)
Filas con incidencia > 0: 263,091 (9.0%)


## 3. Distribución de incidencia

In [11]:
positivos = df.loc[df["incidencia"] > 0, "incidencia"]

fig = make_subplots(rows=1, cols=2, subplot_titles=("Escala lineal", "Escala logarítmica"))

fig.add_trace(
    go.Histogram(x=positivos, nbinsx=100, marker_color="#2a9d8f", name="Lineal"),
    row=1, col=1,
)
fig.add_trace(
    go.Histogram(x=positivos, nbinsx=100, marker_color="#e76f51", name="Log"),
    row=1, col=2,
)
fig.update_yaxes(type="log", row=1, col=2)
fig.update_layout(
    title_text="Distribución de incidencia (valores > 0)",
    showlegend=False, height=400,
)
fig.show()

In [12]:
fig = px.box(
    positivos.to_frame(),
    y="incidencia",
    title="Boxplot de incidencia (valores > 0)",
    labels={"incidencia": "Incidencia"},
    height=400,
)
fig.show()
print(positivos.describe())

count    263091.000000
mean          7.665466
std          26.658795
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max        1571.000000
Name: incidencia, dtype: float64


## 4. Análisis temporal — Incidencia por mes

In [13]:
inc_mes = (
    df.groupby(["mes", "mes_nombre"])["incidencia"]
    .sum()
    .reset_index()
    .sort_values("mes")
)

fig = px.bar(
    inc_mes,
    x="mes_nombre",
    y="incidencia",
    text="incidencia",
    title="Incidencia total por mes — 2025",
    labels={"mes_nombre": "Mes", "incidencia": "Incidencia"},
    color="incidencia",
    color_continuous_scale="YlOrRd",
    category_orders={"mes_nombre": MONTH_ORDER},
)
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(height=450, coloraxis_showscale=False)
fig.show()

## 5. Ranking de entidades

In [14]:
ent = (
    df.groupby("entidad")["incidencia"]
    .sum()
    .sort_values(ascending=True)
    .reset_index()
)

fig = px.bar(
    ent,
    x="incidencia",
    y="entidad",
    orientation="h",
    title="Incidencia total por entidad — 2025",
    labels={"entidad": "", "incidencia": "Incidencia"},
    color="incidencia",
    color_continuous_scale="Viridis",
    text="incidencia",
)
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(height=700, coloraxis_showscale=False)
fig.show()

## 6. Top 20 municipios

In [15]:
top_mun = (
    df.groupby(["cvegeo", "municipio", "entidad"])["incidencia"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)
top_mun["label"] = top_mun["municipio"] + " (" + top_mun["entidad"] + ")"
top_mun = top_mun.sort_values("incidencia", ascending=True)

fig = px.bar(
    top_mun,
    x="incidencia",
    y="label",
    orientation="h",
    title="Top 20 municipios por incidencia — 2025",
    labels={"label": "", "incidencia": "Incidencia"},
    color="entidad",
    text="incidencia",
)
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(height=600, legend_title_text="Entidad")
fig.show()

## 7. Tipos de delito y bien jurídico

In [16]:
top_delito = (
    df.groupby("tipo_delito")["incidencia"]
    .sum()
    .sort_values(ascending=True)
    .tail(15)
    .reset_index()
)

fig = px.bar(
    top_delito,
    x="incidencia",
    y="tipo_delito",
    orientation="h",
    title="Top 15 tipos de delito — 2025",
    labels={"tipo_delito": "", "incidencia": "Incidencia"},
    color="incidencia",
    color_continuous_scale="Reds",
    text="incidencia",
)
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(height=500, coloraxis_showscale=False)
fig.show()

In [17]:
bien_j = (
    df.groupby("bien_juridico")["incidencia"]
    .sum()
    .reset_index()
    .sort_values("incidencia", ascending=False)
)

fig = px.pie(
    bien_j,
    values="incidencia",
    names="bien_juridico",
    title="Distribución por bien jurídico afectado — 2025",
    color_discrete_sequence=px.colors.qualitative.Set2,
    hole=0.35,
)
fig.update_traces(textinfo="percent+label", textposition="outside")
fig.update_layout(height=500, showlegend=False)
fig.show()

## 8. Evolución mensual — Top 5 delitos

In [18]:
top5 = df.groupby("tipo_delito")["incidencia"].sum().nlargest(5).index.tolist()

df_top5 = (
    df[df["tipo_delito"].isin(top5)]
    .groupby(["mes", "tipo_delito"])["incidencia"]
    .sum()
    .reset_index()
)
df_top5["mes_label"] = df_top5["mes"].map(dict(enumerate(MONTH_LABELS, 1)))

fig = px.line(
    df_top5,
    x="mes",
    y="incidencia",
    color="tipo_delito",
    markers=True,
    title="Evolución mensual — Top 5 tipos de delito",
    labels={"mes": "Mes", "incidencia": "Incidencia", "tipo_delito": "Tipo de delito"},
)
fig.update_xaxes(tickvals=list(range(1, 13)), ticktext=MONTH_LABELS)
fig.update_layout(height=450)
fig.show()

## 9. Heatmap — Entidad × Mes

In [19]:
heatmap_data = (
    df.groupby(["entidad", "mes"])["incidencia"]
    .sum()
    .reset_index()
    .pivot(index="entidad", columns="mes", values="incidencia")
)
heatmap_data.columns = MONTH_LABELS

# Ordenar por incidencia total descendente
heatmap_data = heatmap_data.loc[heatmap_data.sum(axis=1).sort_values(ascending=True).index]

fig = px.imshow(
    heatmap_data,
    title="Incidencia por entidad y mes — 2025",
    labels={"x": "Mes", "y": "", "color": "Incidencia"},
    color_continuous_scale="YlOrRd",
    aspect="auto",
    text_auto=",.0f",
)
fig.update_layout(height=800)
fig.show()

## 10. Concentración geográfica (Pareto)

In [20]:
ent_sorted = (
    df.groupby("entidad")["incidencia"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
ent_sorted["pct_acum"] = ent_sorted["incidencia"].cumsum() / ent_sorted["incidencia"].sum() * 100

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=ent_sorted["entidad"],
        y=ent_sorted["incidencia"],
        name="Incidencia",
        marker_color="#2a9d8f",
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=ent_sorted["entidad"],
        y=ent_sorted["pct_acum"],
        name="% Acumulado",
        mode="lines+markers",
        marker_color="#e76f51",
    ),
    secondary_y=True,
)
fig.add_hline(y=80, line_dash="dash", line_color="gray", secondary_y=True, annotation_text="80%")

fig.update_layout(
    title="Pareto — Concentración de incidencia por entidad",
    xaxis_tickangle=-45,
    height=500,
)
fig.update_yaxes(title_text="Incidencia", secondary_y=False)
fig.update_yaxes(title_text="% Acumulado", secondary_y=True, range=[0, 105])
fig.show()

n_80 = (ent_sorted["pct_acum"] <= 80).sum() + 1
print(f"Las primeras {n_80} entidades concentran ~80% de la incidencia total.")

Las primeras 16 entidades concentran ~80% de la incidencia total.


## 11. Homicidios por entidad

In [21]:
hom = (
    df[df["tipo_delito"] == "Homicidio"]
    .groupby("entidad")["incidencia"]
    .sum()
    .sort_values(ascending=True)
    .reset_index()
)

fig = px.bar(
    hom,
    x="incidencia",
    y="entidad",
    orientation="h",
    title="Homicidios por entidad — 2025",
    labels={"entidad": "", "incidencia": "Homicidios"},
    color="incidencia",
    color_continuous_scale="Reds",
    text="incidencia",
)
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(height=700, coloraxis_showscale=False)
fig.show()

## 12. Estacionalidad por tipo de delito

In [22]:
# Heatmap: tipo de delito (top 10) × mes, normalizado por fila para ver estacionalidad
top10_delitos = df.groupby("tipo_delito")["incidencia"].sum().nlargest(10).index.tolist()

season = (
    df[df["tipo_delito"].isin(top10_delitos)]
    .groupby(["tipo_delito", "mes"])["incidencia"]
    .sum()
    .reset_index()
    .pivot(index="tipo_delito", columns="mes", values="incidencia")
)
# Normalizar por fila (% del total anual de cada delito)
season_pct = season.div(season.sum(axis=1), axis=0) * 100
season_pct.columns = MONTH_LABELS

fig = px.imshow(
    season_pct,
    title="Estacionalidad — % mensual por tipo de delito (Top 10)",
    labels={"x": "Mes", "y": "", "color": "% del total anual"},
    color_continuous_scale="RdYlGn_r",
    aspect="auto",
    text_auto=".1f",
)
fig.update_layout(height=450)
fig.show()

## 13. Insights para reporte comercial

### Concentración geográfica
- **10 entidades concentran el 65% de la incidencia total.** Estado de México, CDMX y Guanajuato lideran. Cualquier estrategia de mitigación de riesgo debe priorizar estas zonas.
- **20 municipios acumulan ~29% de todos los delitos.** León, Tijuana, Puebla, Ecatepec y Querétaro son los focos principales. Esto permite focalizar recursos de análisis y monitoreo.

### Tipos de delito
- **Robo es el delito dominante** (474k incidencias), seguido de violencia familiar (267k) y lesiones (212k). Estos tres representan la mayor parte de la carga delictiva.
- **El patrimonio es el bien jurídico más afectado**, seguido de la familia y la integridad corporal.
- **Homicidios:** Guanajuato, Estado de México y Michoacán lideran en cifras absolutas. Guanajuato destaca por su alta tasa relativa.

### Estacionalidad
- La incidencia es **relativamente estable** a lo largo del año, con ligera baja en noviembre-diciembre.
- **Robo** tiene su pico en enero y baja hacia fin de año.
- **Violencia familiar** sube progresivamente y alcanza su máximo en mayo.
- **Lesiones** tienen su pico en marzo.

### Implicaciones comerciales
- **Seguros y fianzas:** La concentración geográfica permite diseñar primas diferenciadas por municipio con alta precisión.
- **Real estate / retail:** Los 20 municipios de mayor riesgo deben tener evaluaciones de seguridad reforzadas antes de abrir nuevas ubicaciones.
- **Logística y transporte:** El robo como delito dominante impacta directamente costos de operación en rutas que cruzan los estados de mayor incidencia.
- **Recursos humanos:** La violencia familiar como segundo delito más frecuente tiene implicaciones en productividad laboral y programas de bienestar corporativo.